# 🧬 Mô Hình Baseline Thuần Văn Bản (TextMLP) Dự Đoán Tương Tác Thuốc (DDI)
### Đánh Giá Toàn Diện Trên 3 Kịch Bản ($S_0, S_1, S_2$) & Tích Hợp Weights & Biases (W&B)

---

### 📌 1. Bối Cảnh & Mục Tiêu Mô Hình Baseline
Notebook này xây dựng mô hình baseline **Multi-Layer Perceptron (TextMLP)** hoàn chỉnh cho bài toán phân loại đa lớp tương tác thuốc (**65 classes DDI**), phục vụ làm cột mốc tham chiếu khoa học (Benchmark).

- **Ràng buộc nghiêm ngặt:** Chỉ sử dụng **dữ liệu văn bản (Biomedical Text Descriptions)** từ [drug_features.json](file:///home/tuannoiuem/entrance_ddi_test/drug_features.json). Tuyệt đối **không** sử dụng cấu trúc phân tử (SMILES) hay mã phân loại Target/Enzyme/Pathway có sẵn.
- **Chống rò rỉ dữ liệu (Anti-Leakage):** Bộ trích xuất TF-IDF + TruncatedSVD được huấn luyện (`fit`) **duy nhất** trên các thuốc trong tập Train (`drug_features_train.json`), sau đó mới `transform` sang các thuốc mới Unseen (`drug_features_unseen.json`).
- **Đánh giá đa chiều:** Đo lường đồng thời **6 chỉ số cốt lõi**:
  1. `Accuracy`
  2. `Macro Precision`
  3. `Macro Recall`
  4. `Macro F1-Score`
  5. `Micro ROC-AUC` (One-vs-Rest)
  6. `Micro AUPR / PR-AUC` (Average Precision - thước đo quan trọng nhất cho dữ liệu mất cân bằng nặng)
- **Tracking thời gian thực:** Đồng bộ toàn bộ quá trình huấn luyện và bảng tổng kết đánh giá lên **Weights & Biases (W&B)**.


In [7]:
import os
import sys
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

import wandb

warnings.filterwarnings('ignore')

# Thiết lập Random Seed cố định để đảm bảo tính tái lập 100%
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cấu hình phong cách biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'

# Đường dẫn tài nguyên
DATA_DIR = Path('/kaggle/input/datasets/tuantc2306/train-data')
print(f"✓ Thiết bị tính toán (Device): {device}")
print(f"✓ Thư mục dữ liệu: {DATA_DIR}")


✓ Thiết bị tính toán (Device): cuda
✓ Thư mục dữ liệu: /kaggle/input/datasets/tuantc2306/train-data


## 1. Nạp Dữ Liệu Tương Tác & Đặc Trưng Thuốc Từ `data2/`
Nạp các tệp dữ liệu cố định:
- `train.csv` (17,276 cặp): Tập huấn luyện.
- `val.csv` (3,502 cặp): Tập validation.
- `test_s0.csv` (3,502 cặp): Kịch bản Transductive (cả 2 thuốc đã biết trong Train).
- `test_s1.csv` (11,602 cặp): Kịch bản Semi-Inductive (1 thuốc đã biết, 1 thuốc mới).
- `test_s2.csv` (1,382 cặp): Kịch bản Fully-Inductive (cả 2 thuốc đều mới).
- `drug_features_train.json` (456 thuốc) & `drug_features_unseen.json` (114 thuốc).


In [8]:
# Tải các bảng dữ liệu tương tác
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_s0_df = pd.read_csv(DATA_DIR / 'test_s0.csv')
test_s1_df = pd.read_csv(DATA_DIR / 'test_s1.csv')
test_s2_df = pd.read_csv(DATA_DIR / 'test_s2.csv')

# Tải đặc trưng thuốc
with open(DATA_DIR / 'drug_features_train.json', 'r', encoding='utf-8') as f:
    drug_features_train = json.load(f)
with open(DATA_DIR / 'drug_features_unseen.json', 'r', encoding='utf-8') as f:
    drug_features_unseen = json.load(f)

print("=== TỔNG QUAN DỮ LIỆU ĐẦU VÀO TỪ DATA2/ ===")
print(f"• Tập Train:   {train_df.shape[0]:>6,} cặp ({train_df['drug1_id'].nunique()} d1, {train_df['drug2_id'].nunique()} d2)")
print(f"• Tập Val:     {val_df.shape[0]:>6,} cặp ({val_df['label'].nunique()} classes)")
print(f"• Tập Test S0: {test_s0_df.shape[0]:>6,} cặp (Seen - Seen)")
print(f"• Tập Test S1: {test_s1_df.shape[0]:>6,} cặp (Seen - Unseen)")
print(f"• Tập Test S2: {test_s2_df.shape[0]:>6,} cặp (Unseen - Unseen)")
print(f"• Thuốc trong Train Features:  {len(drug_features_train):>3} thuốc")
print(f"• Thuốc trong Unseen Features: {len(drug_features_unseen):>3} thuốc")


=== TỔNG QUAN DỮ LIỆU ĐẦU VÀO TỪ DATA2/ ===
• Tập Train:   17,276 cặp (439 d1, 448 d2)
• Tập Val:      3,502 cặp (59 classes)
• Tập Test S0:  3,502 cặp (Seen - Seen)
• Tập Test S1: 11,602 cặp (Seen - Unseen)
• Tập Test S2:  1,382 cặp (Unseen - Unseen)
• Thuốc trong Train Features:  456 thuốc
• Thuốc trong Unseen Features: 114 thuốc


## 2. Trích Xuất Đặc Trưng Văn Bản Y Sinh Chống Rò Rỉ (Anti-Leakage)
Khai thác 5 trường văn bản: `description`, `indication`, `pharmacodynamics`, `mechanism_of_action`, `metabolism`:
- Ghép nối có ngữ cảnh cho từng loại thuốc.
- **Bộ mã hóa TF-IDF + TruncatedSVD (d = 256):** Được `fit` **duy nhất** trên 456 thuốc huấn luyện (`drug_features_train.json`).
- Sau đó dùng mô hình đã fit để `transform` sang 114 thuốc mới (`drug_features_unseen.json`).
- Tuyệt đối bảo đảm không có bất kỳ từ vựng hay thống kê nào từ tập Test bị rò rỉ vào không gian vector.


In [9]:
text_fields = ['description', 'indication', 'pharmacodynamics', 'mechanism_of_action', 'metabolism']

def build_drug_text_corpus(feat_dict):
    """Làm sạch và ghép nối 5 trường văn bản y sinh có ngữ cảnh"""
    corpus = {}
    for d_id, feat in feat_dict.items():
        parts = []
        for field in text_fields:
            val = feat.get(field, '')
            if val and isinstance(val, str) and val.strip():
                clean_val = val.replace('<p>', ' ').replace('</p>', ' ').replace('<i>', ' ').replace('</i>', ' ')
                clean_val = ' '.join(clean_val.split())
                parts.append(f"{field.capitalize()}: {clean_val}")
        corpus[d_id] = ' '.join(parts) if parts else 'Unknown medication'
    return corpus

# Xây dựng corpus văn bản
train_corpus = build_drug_text_corpus(drug_features_train)
unseen_corpus = build_drug_text_corpus(drug_features_unseen)

train_drug_ids = sorted(list(train_corpus.keys()))
unseen_drug_ids = sorted(list(unseen_corpus.keys()))

print("=== TIẾN HÀNH TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN (CHỐNG DATA LEAKAGE) ===")
# 1. Khởi tạo TF-IDF và TruncatedSVD
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
svd_model = TruncatedSVD(n_components=256, random_state=SEED)

# 2. FIT CHỈ TRÊN TẬP THUỐC HUẤN LUYỆN
t0 = time.time()
X_tfidf_train = tfidf_vec.fit_transform([train_corpus[d] for d in train_drug_ids])
X_svd_train = svd_model.fit_transform(X_tfidf_train)

# 3. TRANSFORM SANG TẬP THUỐC MỚI (UNSEEN)
X_tfidf_unseen = tfidf_vec.transform([unseen_corpus[d] for d in unseen_drug_ids])
X_svd_unseen = svd_model.transform(X_tfidf_unseen)
t1 = time.time()

# 4. Lưu vào từ điển embedding hoàn chỉnh
drug_embeddings = {}
for idx, d_id in enumerate(train_drug_ids):
    drug_embeddings[d_id] = X_svd_train[idx]
for idx, d_id in enumerate(unseen_drug_ids):
    drug_embeddings[d_id] = X_svd_unseen[idx]

print(f"✓ Trích xuất đặc trưng văn bản hoàn tất sau {t1 - t0:.2f}s!")
print(f"• Số chiều vector thuốc (d): {X_svd_train.shape[1]} chiều")
print(f"• Tỷ lệ phương sai giải thích được (Explained Variance Ratio): {svd_model.explained_variance_ratio_.sum()*100:.2f}%")
print(f"• Tổng số thuốc có embedding: {len(drug_embeddings)} thuốc")


=== TIẾN HÀNH TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN (CHỐNG DATA LEAKAGE) ===
✓ Trích xuất đặc trưng văn bản hoàn tất sau 1.25s!
• Số chiều vector thuốc (d): 256 chiều
• Tỷ lệ phương sai giải thích được (Explained Variance Ratio): 77.86%
• Tổng số thuốc có embedding: 570 thuốc


## 3. Biểu Diễn Cặp Tương Tác Thuốc (Pairwise Representation) & Dataset Pipeline
Xây dựng vector biểu diễn cặp thuốc 769 chiều có hướng:
$$\mathbf{x}_{(d_1, d_2)} = [\mathbf{u}; \; \mathbf{w}; \; |\mathbf{u} - \mathbf{w}|; \; \cos(\mathbf{u}, \mathbf{w})]$$
- $[\mathbf{u}; \mathbf{w}]$: 512 chiều bảo toàn vai trò Victim/Perpetrator.
- $|\mathbf{u} - \mathbf{w}|$: 256 chiều đo khoảng cách sai lệch dược lý.
- $\cos(\mathbf{u}, \mathbf{w})$: 1 chiều đo độ tương đồng góc ngữ nghĩa.


In [10]:
def create_pair_features(df, emb_dict, dim=256):
    """Tạo ma trận vector biểu diễn cặp tương tác 769 chiều"""
    d1_arr = df['drug1_id'].values
    d2_arr = df['drug2_id'].values
    y = df['label'].values if 'label' in df.columns else None
    
    # Tra cứu embedding cho drug1 và drug2
    u = np.array([emb_dict[d] for d in d1_arr], dtype=np.float32)
    w = np.array([emb_dict[d] for d in d2_arr], dtype=np.float32)
    
    diff = np.abs(u - w)
    
    # Cosine similarity
    norm_u = np.linalg.norm(u, axis=1, keepdims=True) + 1e-8
    norm_w = np.linalg.norm(w, axis=1, keepdims=True) + 1e-8
    cos_sim = np.sum(u * w, axis=1, keepdims=True) / (norm_u * norm_w)
    
    # Ghép nối vector: [u; w; |u - w|; cos(u, w)] -> 256 + 256 + 256 + 1 = 769
    X = np.hstack([u, w, diff, cos_sim])
    return X, y

# PyTorch Dataset
class DDIPairDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

print("=== TẠO MATRIX BIỂU DIỄN CẶP CHO TẤT CẢ CÁC TẬP DỮ LIỆU ===")
t_p0 = time.time()
X_train, y_train = create_pair_features(train_df, drug_embeddings)
X_val, y_val = create_pair_features(val_df, drug_embeddings)
X_s0, y_s0 = create_pair_features(test_s0_df, drug_embeddings)
X_s1, y_s1 = create_pair_features(test_s1_df, drug_embeddings)
X_s2, y_s2 = create_pair_features(test_s2_df, drug_embeddings)
t_p1 = time.time()

# Khởi tạo DataLoader
batch_size = 256
train_loader = DataLoader(DDIPairDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(DDIPairDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_s0_loader = DataLoader(DDIPairDataset(X_s0, y_s0), batch_size=512, shuffle=False)
test_s1_loader = DataLoader(DDIPairDataset(X_s1, y_s1), batch_size=512, shuffle=False)
test_s2_loader = DataLoader(DDIPairDataset(X_s2, y_s2), batch_size=512, shuffle=False)

print(f"✓ Tạo đặc trưng cặp hoàn tất sau {t_p1 - t_p0:.2f}s!")
print(f"• X_train shape:   {X_train.shape}")
print(f"• X_val shape:     {X_val.shape}")
print(f"• X_test_s0 shape: {X_s0.shape}")
print(f"• X_test_s1 shape: {X_s1.shape}")
print(f"• X_test_s2 shape: {X_s2.shape}")


=== TẠO MATRIX BIỂU DIỄN CẶP CHO TẤT CẢ CÁC TẬP DỮ LIỆU ===
✓ Tạo đặc trưng cặp hoàn tất sau 0.16s!
• X_train shape:   (17276, 769)
• X_val shape:     (3502, 769)
• X_test_s0 shape: (3502, 769)
• X_test_s1 shape: (11602, 769)
• X_test_s2 shape: (1382, 769)


## 4. Kiến Trúc Mạng Nơ-ron TextMLP & Tính Trọng Số Lớp
Thiết kế mạng nơ-ron đa tầng với cấu trúc Bottleneck:
- `Linear(769 -> 512)` -> `BatchNorm1d` -> `ReLU` -> `Dropout(0.3)`
- `Linear(512 -> 256)` -> `BatchNorm1d` -> `ReLU` -> `Dropout(0.3)`
- `Linear(256 -> 65)`
Sử dụng trọng số lớp nghịch đảo tần số mẫu (`Class Weights`) để triệt tiêu ảnh hưởng mất cân bằng nhãn.


In [11]:
class TextMLP(nn.Module):
    def __init__(self, input_dim=769, hidden_dim1=512, hidden_dim2=256, num_classes=65, dropout_rate=0.3):
        super(TextMLP, self).__init__()
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.block2 = nn.Sequential(
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.classifier = nn.Linear(hidden_dim2, num_classes)
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

# Tính trọng số nghịch đảo tần số cho 65 classes
class_counts = Counter(y_train)
total_samples = len(y_train)
num_classes = 65

weights = np.zeros(num_classes, dtype=np.float32)
for c in range(num_classes):
    count = class_counts.get(c, 1)
    weights[c] = total_samples / (num_classes * (count ** 0.6))

# Chuẩn hóa trọng số
weights = weights / weights.mean()
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

model = TextMLP(input_dim=769, hidden_dim1=512, hidden_dim2=256, num_classes=65, dropout_rate=0.3).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== KIẾN TRÚC MÔ HÌNH TEXT-MLP ===")
print(model)
print(f"• Tổng số tham số có thể huấn luyện (Trainable Parameters): {total_params:,}")
print(f"• Min class weight: {weights.min():.2f} | Max class weight: {weights.max():.2f}")


=== KIẾN TRÚC MÔ HÌNH TEXT-MLP ===
TextMLP(
  (block1): Sequential(
    (0): Linear(in_features=769, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (classifier): Linear(in_features=256, out_features=65, bias=True)
)
• Tổng số tham số có thể huấn luyện (Trainable Parameters): 543,809
• Min class weight: 0.03 | Max class weight: 3.33


## 5. Hàm Đánh Giá Toàn Diện (6 Chỉ Số: ACC, Prec, Rec, F1, AUC, AUPR)
Hàm `evaluate_model(model, loader, device)` tính toán:
1. `Accuracy`
2. `Macro Precision`
3. `Macro Recall`
4. `Macro F1`
5. `Micro ROC-AUC` (One-vs-Rest, xử lý an toàn cho các nhãn hiếm không xuất hiện)
6. `Micro AUPR` (Average Precision, thước đo cốt lõi cho phân loại mất cân bằng)


In [12]:
def evaluate_model(model, loader, device, num_classes=65):
    """
    Tính toán toàn diện 6 chỉ số đánh giá chuyên sâu
    """
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            logits = model(bx)
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(by.numpy())
            all_probs.extend(probs.cpu().numpy())
            
    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    
    # 1. Các chỉ số nhãn rời rạc
    acc = accuracy_score(y_true, y_pred)
    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # 2. Các chỉ số xác suất liên tục (AUC & AUPR)
    present_classes = np.unique(y_true)
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    
    try:
        micro_auc = roc_auc_score(
            y_true_bin[:, present_classes],
            y_prob[:, present_classes],
            average='micro',
            multi_class='ovr'
        )
    except Exception:
        micro_auc = float('nan')
        
    try:
        micro_aupr = average_precision_score(
            y_true_bin[:, present_classes],
            y_prob[:, present_classes],
            average='micro'
        )
    except Exception:
        micro_aupr = float('nan')
        
    return {
        'acc': acc,
        'macro_prec': macro_prec,
        'macro_rec': macro_rec,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'micro_auc': micro_auc,
        'micro_aupr': micro_aupr,
        'targets': y_true,
        'preds': y_pred,
        'probs': y_prob
    }


## 6. Huấn Luyện Mô Hình & Tracking Thời Gian Thực Trên Weights & Biases (W&B)
Huấn luyện TextMLP qua 15 epochs với:
- Tối ưu hóa: `AdamW` (learning rate = 1e-3, weight decay = 1e-4).
- Learning Rate Scheduler: `ReduceLROnPlateau` giảm tốc độ học khi Macro-F1 chững lại.
- Early Stopping: Giám sát Macro-F1 trên tập `val`, lưu checkpoint tốt nhất vào `best_text_mlp_baseline.pt`.
- Đồng bộ hóa từng epoch lên **Weights & Biases**.


In [13]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("entrance_testing")

In [14]:
wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [15]:
# Khởi tạo Weights & Biases Run
WANDB_PROJECT = "DDI_Entrance_Test"
WANDB_RUN_NAME = "BaseLineModel_MLP"

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": "TextMLP",
        "input_dim": 769,
        "hidden_dim1": 512,
        "hidden_dim2": 256,
        "num_classes": 65,
        "dropout": 0.3,
        "batch_size": batch_size,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "epochs": 15,
        "optimizer": "AdamW",
        "loss": "Class-Weighted CrossEntropyLoss",
        "dataset_train_samples": len(train_df),
        "dataset_val_samples": len(val_df)
    },
    reinit=True
)

# Cấu hình huấn luyện
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs = 15
patience = 4
best_val_f1 = -1.0
best_epoch = -1
checkpoint_path = Path("/kaggle/working/") / 'best_text_mlp_baseline.pt'


print(f"=== BẮT ĐẦU HUẤN LUYỆN TEXT-MLP (TRACKING W&B: {wandb_run.name}) ===")
patience_counter = 0

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    train_preds = []
    train_targets = []
    
    t_ep0 = time.time()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * bx.size(0)
        train_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        train_targets.extend(by.cpu().numpy())
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = accuracy_score(train_targets, train_preds)
    epoch_train_f1 = f1_score(train_targets, train_preds, average='macro', zero_division=0)
    
    # Đánh giá trên tập Validation
    val_loss_run = 0.0
    model.eval()
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            l = criterion(model(bx), by)
            val_loss_run += l.item() * bx.size(0)
    epoch_val_loss = val_loss_run / len(val_loader.dataset)
    val_m = evaluate_model(model, val_loader, device, num_classes=65)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_m['macro_f1'])
    t_ep1 = time.time()

    
    # LOG TRỰC TIẾP LÊN WEIGHTS & BIASES
    wandb.log({
        "epoch": epoch,
        "train/loss": epoch_train_loss,
        "train/acc": epoch_train_acc,
        "train/macro_f1": epoch_train_f1,
        "val/loss": epoch_val_loss,
        "val/acc": val_m['acc'],
        "val/macro_precision": val_m['macro_prec'],
        "val/macro_recall": val_m['macro_rec'],
        "val/macro_f1": val_m['macro_f1'],
        "val/micro_auc": val_m['micro_auc'],
        "val/micro_aupr": val_m['micro_aupr'],
        "learning_rate": current_lr
    })
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] ({t_ep1 - t_ep0:.1f}s) | "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.1f}% F1: {epoch_train_f1:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {val_m['acc']*100:.1f}% F1: {val_m['macro_f1']:.4f} "
          f"Prec: {val_m['macro_prec']:.4f} Rec: {val_m['macro_rec']:.4f} AUC: {val_m['micro_auc']:.4f} AUPR: {val_m['micro_aupr']:.4f}")
    
    # Early Stopping check
    if val_m['macro_f1'] > best_val_f1:
        best_val_f1 = val_m['macro_f1']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n✋ Kích hoạt Early Stopping tại Epoch {epoch}! Khôi phục checkpoint tốt nhất từ Epoch {best_epoch} (Val Macro-F1: {best_val_f1:.4f})")
            break

# Khôi phục model tốt nhất
model.load_state_dict(torch.load(checkpoint_path))
print(f"\n✓ Đã khôi phục checkpoint tốt nhất từ: {checkpoint_path}")


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


=== BẮT ĐẦU HUẤN LUYỆN TEXT-MLP (TRACKING W&B: BaseLineModel_MLP) ===
Epoch [01/15] (1.2s) | Train Loss: 2.7379 Acc: 46.7% F1: 0.2074 | Val Loss: 1.9924 Acc: 60.8% F1: 0.4190 Prec: 0.3801 Rec: 0.5672 AUC: 0.9835 AUPR: 0.5749
Epoch [02/15] (0.5s) | Train Loss: 1.2713 Acc: 66.3% F1: 0.4570 | Val Loss: 1.2250 Acc: 67.6% F1: 0.5770 Prec: 0.4934 Rec: 0.8043 AUC: 0.9887 AUPR: 0.7217
Epoch [03/15] (0.4s) | Train Loss: 0.7674 Acc: 72.7% F1: 0.6101 | Val Loss: 1.0017 Acc: 71.6% F1: 0.5552 Prec: 0.4687 Rec: 0.7727 AUC: 0.9910 AUPR: 0.7742
Epoch [04/15] (0.4s) | Train Loss: 0.5426 Acc: 76.7% F1: 0.6957 | Val Loss: 0.8034 Acc: 76.0% F1: 0.6138 Prec: 0.5386 Rec: 0.7701 AUC: 0.9939 AUPR: 0.8301
Epoch [05/15] (0.4s) | Train Loss: 0.4056 Acc: 80.1% F1: 0.7481 | Val Loss: 0.7550 Acc: 77.6% F1: 0.6450 Prec: 0.5682 Rec: 0.8182 AUC: 0.9940 AUPR: 0.8397
Epoch [06/15] (0.4s) | Train Loss: 0.3409 Acc: 82.4% F1: 0.7922 | Val Loss: 0.6723 Acc: 79.4% F1: 0.6369 Prec: 0.5650 Rec: 0.7920 AUC: 0.9951 AUPR: 0.8642


## 7. Đánh Giá Đối Chiếu Hiệu Năng Trên 3 Kịch Bản ($S_0, S_1, S_2$)
Đánh giá mô hình tốt nhất trên cả 3 tập kiểm thử với trọn bộ 6 chỉ số:
- **`test_s0` (Transductive):** Cả 2 thuốc đều đã biết trong tập huấn luyện.
- **`test_s1` (Semi-Inductive):** 1 thuốc đã biết, 1 thuốc mới (Cold-Start một phía).
- **`test_s2` (Fully-Inductive):** Cả 2 thuốc đều mới hoàn toàn (Zero-Shot).
Đồng bộ bảng tổng kết lên Weights & Biases Dashboard.


In [17]:
# Đánh giá trên 3 tập kiểm thử
test_s0_metrics = evaluate_model(model, test_s0_loader, device, num_classes=65)
test_s1_metrics = evaluate_model(model, test_s1_loader, device, num_classes=65)
test_s2_metrics = evaluate_model(model, test_s2_loader, device, num_classes=65)

# Tạo DataFrame tổng hợp
summary_records = [
    {
        'Kịch Bản Kiểm Thử': 'Test S0 (Transductive)',
        'Bản Chất Cặp Thuốc': 'Cả 2 thuốc đã biết trong Train',
        'Số Mẫu': len(test_s0_df),
        'Accuracy (%)': f"{test_s0_metrics['acc']*100:.2f}%",
        'Macro Prec': f"{test_s0_metrics['macro_prec']:.4f}",
        'Macro Rec': f"{test_s0_metrics['macro_rec']:.4f}",
        'Macro F1': f"{test_s0_metrics['macro_f1']:.4f}",
        'Micro AUC': f"{test_s0_metrics['micro_auc']:.4f}",
        'Micro AUPR': f"{test_s0_metrics['micro_aupr']:.4f}"
    },
    {
        'Kịch Bản Kiểm Thử': 'Test S1 (Semi-Inductive)',
        'Bản Chất Cặp Thuốc': '1 thuốc đã biết, 1 thuốc mới',
        'Số Mẫu': len(test_s1_df),
        'Accuracy (%)': f"{test_s1_metrics['acc']*100:.2f}%",
        'Macro Prec': f"{test_s1_metrics['macro_prec']:.4f}",
        'Macro Rec': f"{test_s1_metrics['macro_rec']:.4f}",
        'Macro F1': f"{test_s1_metrics['macro_f1']:.4f}",
        'Micro AUC': f"{test_s1_metrics['micro_auc']:.4f}",
        'Micro AUPR': f"{test_s1_metrics['micro_aupr']:.4f}"
    },
    {
        'Kịch Bản Kiểm Thử': 'Test S2 (Fully-Inductive)',
        'Bản Chất Cặp Thuốc': 'Cả 2 thuốc đều mới hoàn toàn (Zero-Shot)',
        'Số Mẫu': len(test_s2_df),
        'Accuracy (%)': f"{test_s2_metrics['acc']*100:.2f}%",
        'Macro Prec': f"{test_s2_metrics['macro_prec']:.4f}",
        'Macro Rec': f"{test_s2_metrics['macro_rec']:.4f}",
        'Macro F1': f"{test_s2_metrics['macro_f1']:.4f}",
        'Micro AUC': f"{test_s2_metrics['micro_auc']:.4f}",
        'Micro AUPR': f"{test_s2_metrics['micro_aupr']:.4f}"
    }
]

df_summary = pd.DataFrame(summary_records)
print("=" * 115)
print("BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH BASELINE TEXT-MLP TRÊN 3 KỊCH BẢN (DATA2/)")
print("=" * 115)
print(df_summary.to_string(index=False))
print("=" * 115)

# Lưu kết quả vào tệp CSV
results_csv_path = Path("/kaggle/working/") / 'baseline_results_summary.csv'
df_summary.to_csv(results_csv_path, index=False)
print(f"✓ Đã xuất kết quả đánh giá ra tệp: {results_csv_path}")

# TẠO W&B TABLE VÀ GỬI LÊN W&B
wb_table = wandb.Table(
    columns=["Setting", "Scenario", "Samples", "Accuracy (%)", "Macro Prec", "Macro Rec", "Macro F1", "Micro AUC", "Micro AUPR"]
)
for r in summary_records:
    wb_table.add_data(
        r['Kịch Bản Kiểm Thử'], r['Bản Chất Cặp Thuốc'], r['Số Mẫu'],
        r['Accuracy (%)'], r['Macro Prec'], r['Macro Rec'], r['Macro F1'],
        r['Micro AUC'], r['Micro AUPR']
    )

wandb.log({
    "Final Evaluation Comparison": wb_table,
    "test_s0/macro_f1": test_s0_metrics['macro_f1'],
    "test_s0/micro_auc": test_s0_metrics['micro_auc'],
    "test_s0/micro_aupr": test_s0_metrics['micro_aupr'],
    "test_s1/macro_f1": test_s1_metrics['macro_f1'],
    "test_s1/micro_auc": test_s1_metrics['micro_auc'],
    "test_s1/micro_aupr": test_s1_metrics['micro_aupr'],
    "test_s2/macro_f1": test_s2_metrics['macro_f1'],
    "test_s2/micro_auc": test_s2_metrics['micro_auc'],
    "test_s2/micro_aupr": test_s2_metrics['micro_aupr']
})

wandb.finish()
print("✓ Đã hoàn tất đồng bộ hóa và đóng W&B Run thành công!")


BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH BASELINE TEXT-MLP TRÊN 3 KỊCH BẢN (DATA2/)
        Kịch Bản Kiểm Thử                       Bản Chất Cặp Thuốc  Số Mẫu Accuracy (%) Macro Prec Macro Rec Macro F1 Micro AUC Micro AUPR
   Test S0 (Transductive)           Cả 2 thuốc đã biết trong Train    3502       84.29%     0.7301    0.8050   0.7558    0.9962     0.9045
 Test S1 (Semi-Inductive)             1 thuốc đã biết, 1 thuốc mới   11602       61.52%     0.4757    0.5233   0.4453    0.9735     0.6516
Test S2 (Fully-Inductive) Cả 2 thuốc đều mới hoàn toàn (Zero-Shot)    1382       41.97%     0.2383    0.1071   0.1154    0.9262     0.3795
✓ Đã xuất kết quả đánh giá ra tệp: /kaggle/working/baseline_results_summary.csv


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,██████████▁▁▁▁▁
test_s0/macro_f1,▁
test_s0/micro_auc,▁
test_s0/micro_aupr,▁
test_s1/macro_f1,▁
test_s1/micro_auc,▁
test_s1/micro_aupr,▁
test_s2/macro_f1,▁
test_s2/micro_auc,▁
+11,...


✓ Đã hoàn tất đồng bộ hóa và đóng W&B Run thành công!


### 💡 Tổng Kết & Đánh Giá Mô Hình Baseline Thuần Văn Bản

1. **Hiệu năng thực tế trên 3 kịch bản:**
   - **`Test S0` (Transductive):** Mô hình đạt kết quả tốt nhờ sự quen thuộc của các thực thể thuốc trong tập huấn luyện.
   - **`Test S1` (Semi-Inductive):** Có sự suy giảm khi xuất hiện 1 thuốc mới hoàn toàn, nhưng thông tin ngữ nghĩa văn bản của thuốc mới vẫn giúp mô hình dự đoán được nhóm tương tác tương ứng.
   - **`Test S2` (Fully-Inductive / Zero-Shot):** Là kịch bản khó khăn nhất khi cả 2 thuốc đều chưa từng xuất hiện. Độ chính xác phản ánh trực tiếp năng lực tổng quát hóa của các trường văn bản mô tả cơ chế và chuyển hóa dược lực học.

2. **Các chỉ số Macro AUC & AUPR:**
   - Cung cấp góc nhìn chân thực về khả năng xếp hạng xác suất của mô hình, đặc biệt là `AUPR` không bị ảnh hưởng bởi tỷ lệ áp đảo của các cặp âm tính hoặc các lớp quá đông mẫu.

3. **Toàn bộ tiến trình đã được lưu lại:**
   - Model Weights: [best_text_mlp_baseline.pt](file:///home/tuannoiuem/entrance_ddi_test/best_text_mlp_baseline.pt)
   - Bảng tổng kết: [baseline_results_summary.csv](file:///home/tuannoiuem/entrance_ddi_test/baseline_results_summary.csv)
   - Biểu đồ: [baseline_text_model_visuals.png](file:///home/tuannoiuem/entrance_ddi_test/baseline_text_model_visuals.png)
   - Dashboard W&B: Sẵn sàng xem lại hoặc đồng bộ hóa lên Cloud.
